In [116]:
from scipy.signal import decimate
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, InputLayer
import os
import random
import h5py



In [117]:

def get_dataset_name(file_name_with_dir):
    
    filename_without_dir = file_name_with_dir.split('/')[-1]
    print(filename_without_dir)
    temp = filename_without_dir.split('_')[:-1]
    print(temp)
    dataset_name = "_".join(temp)
    return dataset_name
filename_path="./data/Intra/train/rest_105923_1.h5"
with h5py. File (filename_path , 'r') as f :
    dataset_name = get_dataset_name(filename_path)
    print(dataset_name)
    matrix = f.get(dataset_name)[()]
    print(type(matrix ))
    print(matrix.shape)

rest_105923_1.h5
['rest', '105923']
rest_105923
<class 'numpy.ndarray'>
(248, 35624)


In [118]:
def z_score_normalization(data):
    mean = data.mean(axis=0)
    std = data.std(axis=0)
    return (data - mean) / std

In [119]:
def segment_data(data, label, window_size=500, stride=500):
    segments = []
    labels = []
    for start in range(0, data.shape[1] - window_size + 1, stride):
        end = start + window_size
        segment = data[:, start:end]
        segments.append(segment)
        labels.append(label)
    return segments, labels


In [120]:
import re

def infer_label_from_filename(filename, label_map):
    filename = filename.lower().replace('\\', '/')
    basename = os.path.basename(filename)
    
    for key in label_map:
        if key in basename:
            return label_map[key]
    
    raise ValueError(f"Could not infer label from filename: {filename}")


In [121]:
def load_and_preprocess(filepath, label_map, window_size=500, stride=500, downsample_factor=20):
    filename = filepath.lower()
    task_label = infer_label_from_filename(filepath, label_map)
    # Load data
    with h5py.File(filepath, 'r') as f:
        datasetname = list(f.keys())[0]
        data = f.get(datasetname)[()]  # Shape: (248, 35624)

    
    mean = data.mean(axis=0)
    std = data.std(axis=0)
    data = (data - mean) / (std + 1e-8)  # Avoid division by zero

   
    data = decimate(data, q=downsample_factor, axis=1)
    
    # Segment the data into overlapping windows
    segments = []
    labels = []
    num_timepoints = data.shape[1]

    for start in range(0, num_timepoints - window_size + 1, stride):
        end = start + window_size
        window = data[:, start:end]
        segments.append(window[..., np.newaxis])  # Add channel dimension for CNN
        labels.append(task_label)
    
    return segments, labels

In [122]:
def get_labels(filename):
    if 'rest' in filename:
        return 0
    elif 'math' in filename:
        return 1
    elif 'memory' in filename:
        return 2
    elif 'motor' in filename:
        return 3


In [123]:
def data_generator(filepaths, label_map, batch_size=8):
    while True:  # Infinite generator
        all_segments, all_labels = [], []
        for filepath in filepaths:
            segments, labels = load_and_preprocess(filepath, label_map)
            all_segments.extend(segments)
            all_labels.extend(labels)

        X = np.array(all_segments)
        y = np.array(all_labels)

        indices = np.arange(len(y))
        np.random.shuffle(indices)
        X = X[indices]
        y = y[indices]

        for i in range(0, len(X), batch_size):
            yield X[i:i+batch_size], y[i:i+batch_size]

In [124]:
def build_cnn(input_shape=(248, 500, 1), num_classes=4):
    model = Sequential([
        InputLayer(input_shape=input_shape),

        Conv2D(32, kernel_size=(3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2)),

        Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2)),

        Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2)),

        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

In [125]:
#downsampled_length = 35624 // 20 = 1781
#segments_per_file = (1781 - 500) // 500 + 1 = 3
#total_segments = 32 * 3 = 96
#steps_per_epoch = 96 // 32 = 3

In [126]:
def count_total_segments(filepaths, label_map, window_size=500, stride=500, downsample_factor=20):
    total = 0
    for fp in filepaths:
        segments, _ = load_and_preprocess(fp, label_map, window_size, stride, downsample_factor)
        total += len(segments)
    return total


In [ ]:
batch_size = 8
total_segments = count_total_segments(filepaths, label_map)
steps_per_epoch = total_segments // batch_size
print(total_segments, steps_per_epoch)

96 12


In [133]:
# Set filepaths and label map
data_dir='./data/Intra/train'
filepaths = [
    os.path.normpath(os.path.join(data_dir, fname))
    for fname in os.listdir(data_dir)
    if fname.endswith('.h5')
]
print(filepaths)
label_map = {
    'rest': 0,
    'math': 1,
    'story': 1,
    'story_math': 1,         
    'working_memory': 2,
    'memory': 2,
    'motor': 3
}


# Build model
model = build_cnn()

# Create generators
train_gen = data_generator(filepaths, label_map, batch_size=8)

# Fit model
steps_per_epoch = 12  # based on number of segments

model.fit(train_gen, steps_per_epoch=steps_per_epoch, epochs=10)


['data\\Intra\\train\\rest_105923_1.h5', 'data\\Intra\\train\\rest_105923_2.h5', 'data\\Intra\\train\\rest_105923_3.h5', 'data\\Intra\\train\\rest_105923_4.h5', 'data\\Intra\\train\\rest_105923_5.h5', 'data\\Intra\\train\\rest_105923_6.h5', 'data\\Intra\\train\\rest_105923_7.h5', 'data\\Intra\\train\\rest_105923_8.h5', 'data\\Intra\\train\\task_motor_105923_1.h5', 'data\\Intra\\train\\task_motor_105923_2.h5', 'data\\Intra\\train\\task_motor_105923_3.h5', 'data\\Intra\\train\\task_motor_105923_4.h5', 'data\\Intra\\train\\task_motor_105923_5.h5', 'data\\Intra\\train\\task_motor_105923_6.h5', 'data\\Intra\\train\\task_motor_105923_7.h5', 'data\\Intra\\train\\task_motor_105923_8.h5', 'data\\Intra\\train\\task_story_math_105923_1.h5', 'data\\Intra\\train\\task_story_math_105923_2.h5', 'data\\Intra\\train\\task_story_math_105923_3.h5', 'data\\Intra\\train\\task_story_math_105923_4.h5', 'data\\Intra\\train\\task_story_math_105923_5.h5', 'data\\Intra\\train\\task_story_math_105923_6.h5', 'data

In [ ]:
segments, labels = load_and_preprocess(filepaths[0], label_map, window_size=500, stride=500)
print("Segments from one file:", len(segments) ,len(filepaths))

Segments from one file: 3 32


In [ ]:
def load_test_data(test_filepaths, label_map, window_size=500, stride=500, downsample_factor=20):
    all_segments = []
    all_labels = []

    for filepath in test_filepaths:
        segments, labels = load_and_preprocess(
            filepath, label_map,
            window_size=window_size,
            stride=stride,
            downsample_factor=downsample_factor
        )
        all_segments.extend(segments)
        all_labels.extend(labels)

    X_test = np.array(all_segments)
    y_test = np.array(all_labels)
    return X_test, y_test


In [ ]:
import glob

# Collect test files
test_folder = './data/Intra/test'  # or ./data/Cross/test1, etc.
#train_filepaths = sorted(glob.glob('./data/Intra/train/*.h5'))
#test_filepaths = sorted(glob.glob('./data/Intra/test/*.h5'))
test_filepaths = glob.glob(os.path.join(test_folder, '*.h5'))
test_filepaths = [os.path.normpath(p) for p in test_filepaths]

# Load and preprocess test data
X_test, y_test = load_test_data(test_filepaths, label_map)
# Option A: Direct evaluation
loss, accuracy = model.evaluate(X_test, y_test, verbose=1)
print(f"Test Accuracy: {accuracy:.4f}")


1/1 [==============================] - 1s 663ms/step - loss: 9.3724 - accuracy: 0.2500
Test Accuracy: 0.2500


In [ ]:
print(y_test)

[0 0 0 0 0 0 3 3 3 3 3 3 1 1 1 1 1 1 2 2 2 2 2 2]
